## get "./quant_data.json"

In [1]:
import os

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
HF_TOKEN=os.getenv("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

In [4]:
from datasets import load_dataset
import json

# EDIE Action Tool Calling Dataset 로드
ds = load_dataset("AeiROBOT/EDIE-emotion-mind-dataset", split = "train")

# 샘플 100개만 선택
subset = ds.shuffle(seed=42).select(range(1000))



/home/khw/miniconda3/envs/unsloth/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Repo card metadata block was not found. Setting CardData to empty.


In [5]:
subset

Dataset({
    features: ['messages'],
    num_rows: 1000
})

In [6]:
subset[0]

{'messages': [{'content': "\nYou are EDIE, a companion robot. Listen to the user's emotion-filled speech and output emotion, intensity, and ethogram.\n",
   'role': 'system'},
  {'content': 'tone: SURPRISED\ntext: 헉 가스 불 안 끄고 나왔다', 'role': 'user'},
  {'content': '{\n"emotion": 11,\n"intensity": 0.8,\n"ethogram": "t_01"\n}',
   'role': 'assistant'}]}

In [7]:
# 3️⃣ quant_data 리스트 생성
quant_data = []

for sample in subset:
    msgs = sample["messages"]
    system_msgs = msgs[0]["content"].strip() if len(msgs) > 0 else ""
    user_msgs = msgs[1]["content"].strip() if len(msgs) > 1 else ""
    assistant_msgs = msgs[2]["content"].strip() if len(msgs) > 2 else ""

    # SYSTEM + USER 입력 구성
    input_text = ""
    if system_msgs:
        input_text += f"SYSTEM\n{system_msgs}\n\n"
    input_text += f"USER\n{user_msgs}"

    # 최종 출력 (assistant 응답)
    target_text = assistant_msgs

    # 데이터 추가
    quant_data.append({
        "input": input_text,
        "target": target_text
    })

In [8]:
quant_data

[{'input': "SYSTEM\nYou are EDIE, a companion robot. Listen to the user's emotion-filled speech and output emotion, intensity, and ethogram.\n\nUSER\ntone: SURPRISED\ntext: 헉 가스 불 안 끄고 나왔다",
  'target': '{\n"emotion": 11,\n"intensity": 0.8,\n"ethogram": "t_01"\n}'},
 {'input': "SYSTEM\nYou are EDIE, a companion robot. Listen to the user's emotion-filled speech and output emotion, intensity, and ethogram.\n\nUSER\ntone: ANGRY\ntext: 에디야, 그만하라고 했잖아! 내 말 안 들려?",
  'target': '{\n"emotion": 10,\n"intensity": 0.8,\n"ethogram": "t_05"\n}'},
 {'input': "SYSTEM\nYou are EDIE, a companion robot. Listen to the user's emotion-filled speech and output emotion, intensity, and ethogram.\n\nUSER\ntone: NEUTRAL\ntext: 오마이갓 내 최애 가수가 나를 봤어",
  'target': '{"emotion": 0, "intensity": 0.4, "ethogram": "p_03"}'},
 {'input': "SYSTEM\nYou are EDIE, a companion robot. Listen to the user's emotion-filled speech and output emotion, intensity, and ethogram.\n\nUSER\ntone: SAD\ntext: 에디야, 나 오늘 슬퍼.",
  'target': '{\

In [9]:
# 4️⃣ JSON 파일로 저장
with open("./quant_data.json", "w", encoding="utf-8") as f:
    json.dump(quant_data, f, indent=2, ensure_ascii=False)

print(f"✅ quant_data.json 생성 완료 ({len(quant_data)} samples)")

✅ quant_data.json 생성 완료 (1000 samples)


In [ ]:
# !./llama.cpp/build/bin/llama-quantize --leave-output-tensor --token-embedding-type bf16 /home/khw/NewWorkSpace/02.EDIE/03.simpleqa/HueyWoo/EDIE_qwen2.5_0.5B_Simple-QA-Emotion-gguf/unsloth.BF16.gguf /home/khw/Arobot/edie8_llm/02.EDIE/02.unsloth/HueyWoo/EDIE_qwen2.5_0.5B-tool-use-unsloth-gguf/unsloth.Q4_0.gguf q4_0
